# Analyser les ouvertures Lichess étape par étape

[Ouvrir dans Colab](https://colab.research.google.com/github/Hadrienfonrojet/ANALYSE-MARCHE-/blob/main/Analyse_data.ipynb)

Le notebook suit ce plan : **1 créer les DataFrames utiles ; 2 coder le χ² ; 3 l’appliquer aux questions annoncées ; 4 Fisher exact, régression multiple et loi F ; 5 descriptifs et graphiques simples.** Les commentaires expliquent chaque colonne. Les résultats sont des associations et restent exploratoires, car un joueur peut contribuer plusieurs parties.


In [ ]:
# 1. Charger une copie fraîche du projet dans Colab et installer les dépendances.
import os
import sys
import subprocess
import tempfile
from pathlib import Path

try:
    import google.colab
except ImportError:
    # En local, ouvrir ce notebook à la racine de la copie du projet à utiliser.
    if not Path("Master_file.py").is_file():
        raise RuntimeError("Ouvrir le notebook dans le dossier du projet.")
else:
    # Un nouveau dossier évite de réutiliser silencieusement un ancien clone.
    # Les anciennes collectes et éventuelles modifications restent conservées.
    target = (
        Path(tempfile.mkdtemp(prefix="lichess_reference_", dir="/content")) / "projet"
    )
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/Hadrienfonrojet/ANALYSE-MARCHE-.git",
            str(target),
        ],
        check=True,
    )
    os.chdir(target)

sys.path.insert(0, str(Path.cwd()))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True
)
print("Projet chargé :", Path.cwd())
print(
    "Avant de changer de version, redémarrer le noyau pour recharger les modules Python."
)


## Partie 1A Choisir les paramètres et obtenir une base brute
Une graine est un pseudo de départ. RAW_FILE réutilise une base déjà collectée. Commencer par un petit essai pour vérifier le fonctionnement, puis augmenter l’échantillon avant de discuter les tests. Cinq coups complets représentent dix demi-coups ; le référentiel contient toutes les lignes officielles a.tsv à e.tsv, et non seulement les ouvertures observées.


In [ ]:
SEEDS = ["yuuki-asuna"]
USERS_PER_SEED = 1
GAMES_PER_FORMAT = 20
RAW_FILE = None  # Ou chemin vers games_enrichies_raw.parquet
OPENING_MOVES = 10  # 5 ou 10 coups complets
RUN_REGRESSIONS = True


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
import config_projet as cfg
from Scripts_generation_data.Import_des_users import discover_stratified
from Scripts_generation_data.Import_une_partie import import_games_enriched

cfg.SEED_USERS = [name.strip().lower() for name in SEEDS]
run_dir = (
    cfg.create_run()
)  ## Créer un nouveau dossier, sans effacer les anciennes bases.

if RAW_FILE is None:
    token = cfg.get_lichess_token()
    users = discover_stratified(cfg.SEED_USERS, token, USERS_PER_SEED)
    raw = import_games_enriched(users, token, GAMES_PER_FORMAT)
else:
    raw = pd.read_parquet(RAW_FILE)  ## Un DataFrame : une ligne par joueur/partie.
    raw.to_parquet(cfg.DATA_DIR / "games_enrichies_raw.parquet", index=False)

print("Lignes :", len(raw), "Joueurs :", raw.user_id.nunique())
display(raw.head())


## Partie 1B Ouvertures et nettoyage
`build_openings()` réunit les 3865 lignes officielles de la version figée. `prepare()` rejoue les positions des premiers coups et cherche la variante la plus précise atteinte dans la fenêtre. Certaines transpositions sont reconnues. Le nom API reste dans `api_opening_name` pour comparaison. Les parties trop courtes, coups absents ou invalides ne sont pas réétiquetés silencieusement par l’API.

Le nettoyage retire les identifiants absents et doublons, calcule score et Elo, puis les colonnes temporelles. Le fichier brut n’est pas modifié. Score : victoire 1, nulle 0,5, défaite 0. `win` vaut 1 seulement pour une victoire.


In [ ]:
from Scripts_generation_data.Creation_openings_complet import build_openings
from Nettoyage_donnees import prepare

openings = build_openings()
clean = prepare(raw, openings, max_fullmoves=OPENING_MOVES)

display(openings[["eco", "name", "pgn", "reference_ply"]].head())
display(
    clean[
        [
            "user_id",
            "opening_name",
            "api_opening_name",
            "opening_reference_ply",
            "opening_identification_status",
        ]
    ].head()
)
display(clean["opening_identification_status"].value_counts())


## Partie 1C Créer les colonnes de passé
Une affectation `d['colonne'] = ...` crée une **colonne** dans le DataFrame d. `groupby(['user_id','color'])` rassemble les lignes du même joueur et de la même couleur pour le calcul ; `shift(1)` lit la valeur de la ligne précédente de ce groupe. Il ne crée pas une liste des parties ayant la même ouverture.

`prev_opening_same_color` contient donc, à chaque ligne, la famille de la précédente partie observée du même joueur avec la même couleur. `repeat_same_color` la compare à la famille actuelle. La première partie d’une couleur est manquante. Les historiques sont limités à la collecte ; des parties réellement jouées peuvent manquer entre deux lignes.


In [ ]:
def add_transitions(frame):
    """Créer des COLONNES d'historique dans un DataFrame trié.

    Une ligne reste une observation joueur/partie. groupby ne fabrique pas
    une base distincte par joueur : il délimite les groupes pour shift(1).
    Une ouverture inconnue ne vaut ni répétition ni changement.
    """
    d = frame.sort_values(["user_id", "timestamp", "game_id"]).copy()
    ## On crée un nouveau DataFrame d, sans modifier le DataFrame fourni.
    d["known_opening"] = d["opening_family"].notna() & ~d["opening_family"].isin(
        ["Inconnue", "", "UNKNOWN"]
    )
    ## La colonne ci-dessus indique si l'ouverture actuelle est connue.

    ## Partie 1A - Lire la partie précédente du même joueur, toute couleur
    by_user = d.groupby("user_id", observed=True)
    d["prev_game_all"] = by_user["game_id"].shift(1)
    d["prev_opening_all"] = by_user["opening_family"].shift(1)
    ## On crée UNE COLONNE : à chaque ligne, la famille de la précédente
    ## partie observée de ce joueur. Ce n'est pas une liste de ses parties.
    d["prev_result_all"] = by_user["result"].shift(1)
    valid_all = d["known_opening"] & by_user["known_opening"].shift(1).eq(True)
    d["repeat_all"] = np.where(
        valid_all, d["opening_family"].eq(d["prev_opening_all"]).astype(float), np.nan
    )
    ## 1 = même famille ; 0 = changement ; NaN = comparaison impossible.

    ## Partie 1B - Lire la précédente partie de même couleur du même joueur
    by_user_color = d.groupby(["user_id", "color"], observed=True)
    d["prev_game_same_color"] = by_user_color["game_id"].shift(1)
    d["prev_opening_same_color"] = by_user_color["opening_family"].shift(1)
    ## On crée UNE COLONNE contenant l'ouverture de la précédente partie
    ## du même joueur AVEC LA MÊME COULEUR. Des parties de l'autre couleur
    ## peuvent s'intercaler. La première partie de chaque groupe vaut NaN.
    d["prev_result_same_color"] = by_user_color["result"].shift(1)
    valid_color = d["known_opening"] & by_user_color["known_opening"].shift(1).eq(True)
    d["repeat_same_color"] = np.where(
        valid_color,
        d["opening_family"].eq(d["prev_opening_same_color"]).astype(float),
        np.nan,
    )

    ## Partie 1C - La couleur elle-même est-elle répétée ?
    d["prev_color"] = by_user["color"].shift(1)
    d["same_color_consecutive"] = np.where(
        d["prev_color"].notna(), d["color"].eq(d["prev_color"]).astype(float), np.nan
    )
    ## Cela ne signifie pas que l'ouverture a été répétée.
    return d


In [ ]:
example = pd.DataFrame(
    {
        "user_id": ["exemple"] * 4,
        "game_id": ["e1", "e2", "e3", "e4"],
        "timestamp": pd.date_range("2024-01-01", periods=4, tz="UTC"),
        "color": ["white", "black", "white", "white"],
        "opening_family": ["A", "B", "A", "B"],
        "result": ["Win", "Loss", "Draw", "Win"],
    }
)
example = add_transitions(example)
display(
    example[
        [
            "game_id",
            "color",
            "opening_family",
            "prev_opening_same_color",
            "repeat_all",
            "repeat_same_color",
        ]
    ]
)
clean = add_transitions(clean)


## Partie 1D Créer les DataFrames nécessaires
Ici, `.dropna()` sélectionne des lignes et `.copy()` crée une base séparée. On garde une base complète pour les descriptifs, une base avec une précédente partie connue toutes couleurs et une base avec une précédente partie connue de même couleur. Les tailles peuvent différer : il faut lire les dénominateurs.


In [ ]:
def creer_dataframes(data):
    """Une base complète et deux sous-bases avec un historique disponible.

    La sélection de lignes par .dropna crée un DataFrame, alors qu'une
    affectation data['nouvelle_colonne'] crée une colonne dans la base existante.
    Les ouvertures inconnues ne permettent pas de comparer les familles.
    """
    parties = add_transitions(data)  ## Copie triée contenant les colonnes de passé.
    toutes_couleurs = parties.dropna(subset=["repeat_all", "prev_result_all"]).copy()
    meme_couleur = parties.dropna(
        subset=["repeat_same_color", "prev_result_same_color"]
    ).copy()
    ## Chaque ligne de meme_couleur peut être comparée à une partie antérieure
    ## du même joueur, avec la même couleur et une ouverture connue.
    return {
        "parties": parties,
        "toutes_couleurs": toutes_couleurs,
        "meme_couleur": meme_couleur,
    }


In [ ]:
frames = creer_dataframes(clean)
for name, frame in frames.items():
    print(name, ":", len(frame), "observations")


## Partie 2 Fonction du chi carré
H0 : les deux variables catégorielles sont indépendantes. H1 : elles sont associées. Les cases contiennent des effectifs et non des moyennes ou des pourcentages.

Attendu = total de ligne × total de colonne / n. Statistique χ² = somme de (observé−attendu)² / attendu. Degrés de liberté = (lignes−1)(colonnes−1). Sous H0 et les hypothèses du test, la statistique suit approximativement une loi du χ². La valeur p est l’aire à droite de la statistique dans cette loi ; elle n’est pas la probabilité que H0 soit vraie.

La règle du projet est que toutes les attentes soient au moins 5. `valid=True` ne contrôle que ce point, pas l’indépendance. Le V de Cramér indique la taille de l’association sans signe. La correction de Yates est désactivée pour correspondre à la formule présentée.


In [ ]:
from scipy.stats import chi2_contingency, fisher_exact, chi2, f
from statsmodels.stats.multitest import multipletests


def chi2_details(frame, a, b, label, min_expected=5):
    """Fonction du chi carré : calculer puis expliquer chaque résultat.

    H0 : les variables a et b sont indépendantes. H1 : elles sont associées.
    On compte les observations, puis on compare aux attentes sous H0 :
        Eij = total_ligne_i * total_colonne_j / n
        chi2 = somme((Oij - Eij)**2 / Eij)
        ddl = (nombre_lignes - 1) * (nombre_colonnes - 1)
    La statistique suit approximativement une loi du chi carré sous H0,
    avec les hypothèses du test et des effectifs suffisants.
    La valeur p n'est PAS la probabilité que H0 soit vraie.
    """
    complete = frame[[a, b]].dropna()  ## Nouveau DataFrame des deux colonnes connues.
    observed = pd.crosstab(complete[a], complete[b])  ## Tableau d'EFFECTIFS entiers.
    observed = observed.loc[observed.sum(axis=1).gt(0), observed.sum(axis=0).gt(0)]
    n = int(observed.to_numpy().sum())
    result = dict(
        test=label,
        n=n,
        chi2=np.nan,
        ddl=np.nan,
        p=np.nan,
        cramer_v=np.nan,
        expected_lt5=np.nan,
        min_expected=np.nan,
        valid=False,
        independence_assumed=True,
        interpretation="exploratoire",
        fisher_p=np.nan,
        status="tableau insuffisant",
    )
    expected = pd.DataFrame(index=observed.index, columns=observed.columns, dtype=float)
    residuals = expected.copy()
    if min(observed.shape, default=0) < 2:
        return result, observed, expected, residuals

    ## Partie 2A - Le calcul de Pearson, sans correction de Yates
    stat, p, ddl, exp = chi2_contingency(observed, correction=False)
    expected = pd.DataFrame(exp, index=observed.index, columns=observed.columns)
    residuals = (observed - expected) / np.sqrt(expected)
    ## Les résidus décrivent les cases qui s'écartent des attentes, sans test par case.

    ## Partie 2B - Taille de l'association et diagnostic des petits effectifs
    v = np.sqrt(stat / (n * min(observed.shape[0] - 1, observed.shape[1] - 1)))
    enough = bool((exp >= min_expected).all())
    result.update(
        chi2=float(stat),
        p=float(p),
        ddl=int(ddl),
        cramer_v=float(v),
        expected_lt5=int((exp < min_expected).sum()),
        min_expected=float(exp.min()),
        valid=enough,
        status="effectifs suffisants" if enough else "effectifs faibles",
    )
    ## valid valide seulement les effectifs attendus, PAS l'indépendance des parties.
    if observed.shape == (2, 2):
        result["fisher_p"] = float(
            fisher_exact(observed.to_numpy(), alternative="two-sided")[1]
        )
    return result, observed, expected, residuals


## Partie 3A Application du χ² au résultat précédent
Question : la répétition d’ouverture est-elle associée au résultat précédent du joueur ? On compare victoire, nulle et défaite. Pour l’historique de même couleur, le résultat précédent est lui aussi celui de la précédente partie de même couleur. Ce test ne prouve pas une réaction psychologique à une victoire ou à une défaite.


In [ ]:
summary, observed, expected, residuals = chi2_details(
    frames["meme_couleur"],
    "prev_result_same_color",
    "repeat_same_color",
    "resultat_precedent_repetition",
)
display(pd.Series(summary))
print("Observés")
display(observed)
print("Attendus sous H0")
display(expected)
print("Résidus descriptifs")
display(residuals)


## Partie 3B Application du χ² au résultat courant
Question : les proportions victoire/nulle/défaite diffèrent-elles entre répétition et changement ? C’est une comparaison de distribution des résultats, pas une estimation causale de l’effet de la répétition. Les effectifs et les proportions doivent être montrés ensemble.


In [ ]:
performance, table, exp, resid = chi2_details(
    frames["meme_couleur"], "repeat_same_color", "result", "repetition_resultat_courant"
)
display(pd.Series(performance))
display(table)
display(table.div(table.sum(axis=1), axis=0))


## Partie 3C Variantes selon la couleur
La batterie principale compte dix comparaisons : les deux questions précédentes selon quatre définitions (toutes couleurs, précédente de même couleur, consécutives observées de même couleur ou de couleur différente), puis deux questions sur la répétition de la couleur elle-même. Ce sont des populations qui se recouvrent, pas quatre échantillons indépendants.

Holm corrige les χ² de cette batterie dont les attentes sont suffisantes. La correction ne répare pas la dépendance des parties. Les stratifications nombreuses et le filtrage d’une transition par joueur restent des options avancées désactivées ici. Le bootstrap descriptif exporté par la fonction reste un complément facultatif ; les conclusions principales reposent sur les tableaux.


In [ ]:
from analyses.Tests_repetition import run_repetition_tests

tests_repetition, tables, taux, bootstrap = run_repetition_tests(
    clean,
    cfg.RESULTS_DIR / "repetition",
    stratify=False,
    include_sensitivity=False,
)
display(
    tests_repetition[
        ["test", "n", "chi2", "ddl", "p", "p_holm", "cramer_v", "min_expected", "valid"]
    ]
)
display(taux)


### Voir la loi théorique du χ²
La courbe suivante représente une loi théorique, pas la distribution des Elo. L’aire colorée à droite de la statistique correspond à la valeur p approximative, si les hypothèses sont acceptables. Si le tableau ne peut pas être testé, la cellule ne trace rien.


In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

if pd.notna(performance["p"]):
    x = np.linspace(0.001, max(15, performance["chi2"] + 4), 400)
    density = chi2.pdf(x, performance["ddl"])
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(x, density)
    ax.axvline(performance["chi2"], color="darkorange")
    ax.fill_between(x, density, where=x >= performance["chi2"], alpha=0.2)
    ax.set(
        title="Loi théorique du chi carré", xlabel="Statistique χ²", ylabel="Densité"
    )
    fig.tight_layout()
    fig.savefig(cfg.RESULTS_DIR / "loi_chi2.png", dpi=150)
    display(fig)
    plt.close(fig)


## Partie 4A Fonction du test exact de Fisher
Le test exact de Fisher utilise un tableau 2×2 et une loi hypergéométrique à marges fixées. H0 : odds ratio de population égal à 1. On choisit un test bilatéral avant de lire les résultats. Il est utile pour de faibles effectifs mais ne corrige pas la dépendance entre joueurs ou parties.

Pour obtenir deux catégories de résultats, on regroupe défaites et nulles dans non-victoire. Ce n’est pas la même question que le χ² victoire/nulle/défaite. Il ne s’agit PAS du test F de régression expliqué ensuite.


In [ ]:
def fisher_details(frame, a, b, label):
    """Comparer deux variables à deux catégories dans une table 2 × 2.

    H0 : odds ratio de population égal à 1 dans un modèle à marges fixées.
    Le calcul exact repose sur une loi hypergéométrique. Il est utile quand
    les effectifs du chi carré sont petits ; il ne corrige pas la dépendance
    entre parties. Ce test ne traite pas plusieurs covariables de régression.
    """
    table = pd.crosstab(frame[a], frame[b])
    table = table.loc[table.sum(axis=1).gt(0), table.sum(axis=0).gt(0)]
    result = dict(
        test=label,
        n=int(table.to_numpy().sum()),
        odds_ratio=np.nan,
        p=np.nan,
        p_holm=np.nan,
        statut="tableau non 2x2",
    )
    if table.shape != (2, 2):
        return result, table
    odds, p_value = fisher_exact(table.to_numpy(), alternative="two-sided")
    result.update(odds_ratio=float(odds), p=float(p_value), statut="exploratoire")
    return result, table


## Partie 4B Appliquer Fisher et lire le tableau
Avec lignes 0=changement, 1=répétition et colonnes 0=non-victoire, 1=victoire, l’odds ratio compare les odds de victoire en répétition à celles en changement. Un odds ratio n’est pas une différence de probabilités. Si une catégorie manque, le test n’est pas calculé. Zéro et infini sont possibles quand une case est vide.


In [ ]:
fisher_summary, fisher_table = fisher_details(
    frames["meme_couleur"], "repeat_same_color", "win", "repetition_victoire"
)
display(fisher_table)
display(pd.Series(fisher_summary))

from analyses.Tests_repetition import run_fisher_tests

fisher_tests, fisher_tables = run_fisher_tests(clean, cfg.RESULTS_DIR / "fisher")
display(fisher_tests)


## Partie 4C Préparer une régression avec plusieurs variables
On explique le score (0 ; 0,5 ; 1) par la répétition de même couleur, l’écart Elo par 100 points, la couleur et la cadence. `C(format_partie)` crée des indicatrices de cadence. C’est un seul modèle linéaire descriptif pour apprendre à tenir plusieurs variables simultanément.

Le coefficient de répétition est une différence ajustée de score dans ce modèle. Il ne mesure pas un effet causal. Les scores sont discrets et bornés ; les prédictions linéaires peuvent sortir de [0,1]. Le modèle ne contrôle pas tous les facteurs, notamment les adversaires communs ou les préférences individuelles.


In [ ]:
import statsmodels.formula.api as smf


def prepare_regression(data):
    """Garder les observations dont les variables nécessaires sont connues."""
    columns = [
        "score",
        "repeat_same_color",
        "elo_diff",
        "is_white",
        "format_partie",
        "user_id",
    ]
    missing = set(columns) - set(data.columns)
    if missing:
        raise ValueError(
            "Colonnes de régression absentes : " + ", ".join(sorted(missing))
        )

    sample = data[columns].copy()  ## Nouveau DataFrame limité aux colonnes du modèle.
    for column in ["score", "repeat_same_color", "elo_diff", "is_white"]:
        sample[column] = pd.to_numeric(sample[column], errors="coerce")
    sample = sample.replace([np.inf, -np.inf], np.nan).dropna()
    sample["elo_diff_100"] = sample["elo_diff"] / 100  ## Une unité représente 100 Elo.
    return sample


In [ ]:
sample = prepare_regression(clean)
print("Lignes complètes pour le modèle :", len(sample))
display(sample.head())
formula = "score ~ repeat_same_color + elo_diff_100 + is_white + C(format_partie)"
print("Formule :", formula)


## Partie 4D Loi F de Fisher Snedecor et test global
H0 : tous les coefficients, sauf la constante, sont nuls. Le F classique compare l’amélioration de la somme des carrés à la variance résiduelle. Il suit exactement une loi F sous les hypothèses classiques d’erreurs normales, indépendantes et de variance constante. Ces hypothèses ne sont pas garanties dans notre panel : le F classique est présenté comme illustration.

La fonction ci-dessous montre le calcul et les degrés de liberté. Le programme propose aussi un F de Wald avec covariance groupée par joueur quand au moins 20 joueurs et assez de groupes sont disponibles. C’est une approximation, pas une loi F exacte en petit échantillon ; les adversaires communs restent une limite. En cas de variable constante, colinéarité ou effectif insuffisant, on affiche un diagnostic sans inventer de résultat.


In [ ]:
def fisher_global_ols(model):
    """Comparer le modèle avec variables au modèle contenant seulement la moyenne.

    H0 : tous les coefficients sauf la constante sont nuls.
    F = ((SCT - SCR) / q) / (SCR / ddl_residus).
    SCT est la somme des carrés autour de la moyenne ; SCR celle des résidus.

    La loi F(q, ddl_residus) est exacte avec erreurs normales, indépendantes et
    de variance constante. Ces conditions ne sont pas garanties pour les scores
    répétés de joueurs : la valeur p classique est un repère pédagogique.
    """
    q = float(model.df_model)
    residual_degrees = float(model.df_resid)
    residual_sum = float(model.ssr)

    if q <= 0 or residual_degrees <= 0 or residual_sum <= 0:
        return {
            "F_classique": np.nan,
            "p_F_classique": np.nan,
            "ddl_numerateur": q,
            "ddl_denominateur": residual_degrees,
        }

    explained_sum = max(0.0, float(model.centered_tss) - residual_sum)
    statistic = (explained_sum / q) / (residual_sum / residual_degrees)
    p_value = f.sf(statistic, q, residual_degrees)  ## Aire à droite dans la loi F.

    return {
        "F_classique": statistic,
        "p_F_classique": p_value,
        "ddl_numerateur": q,
        "ddl_denominateur": residual_degrees,
    }


In [ ]:
def regression_multiple(data):
    """Associer le score à répétition, écart Elo, couleur et cadence.

    beta de repeat_same_color : différence de score ajustée entre répétition
    et changement, à écart Elo, couleur et cadence comparables dans ce modèle.
    Ce coefficient n'est pas un effet causal.

    La constante, les coefficients, le R² et le F classique sont descriptifs.
    Si au moins 20 joueurs et assez de groupes sont disponibles, on fournit
    aussi des erreurs-types groupées par joueur et un F de Wald approximatif.
    Vingt est un seuil de prudence du projet, pas une garantie statistique.
    """
    sample = prepare_regression(data)
    formula = "score ~ repeat_same_color + elo_diff_100 + is_white + C(format_partie)"
    info = {
        "modele": formula,
        "n": len(sample),
        "joueurs": sample["user_id"].nunique(),
        "statut": "non_estimable",
        "raison": "",
        "r2": np.nan,
        "F_classique": np.nan,
        "p_F_classique": np.nan,
        "F_wald_groupes": np.nan,
        "p_F_wald_groupes": np.nan,
        "ddl_numerateur": np.nan,
        "ddl_denominateur": np.nan,
    }
    empty = pd.DataFrame(
        columns=["variable", "coefficient", "ecart_type_groupe", "p_groupe"]
    )

    if len(sample) < 30 or sample["score"].nunique() < 2:
        info["raison"] = (
            "Au moins 30 lignes et une variation du score sont nécessaires."
        )
        return pd.DataFrame([info]), empty, None

    model = smf.ols(formula, data=sample).fit()
    matrix = model.model.exog  ## Matrice numérique créée à partir de la formule.
    n_parameters = matrix.shape[1]

    # Si une variable est constante ou redondante, ne pas prétendre l'estimer.
    if np.linalg.matrix_rank(matrix) < n_parameters or len(sample) < 5 * n_parameters:
        info["raison"] = "Variables redondantes ou trop peu de lignes pour ce modèle."
        return pd.DataFrame([info]), empty, None

    info.update(fisher_global_ols(model))
    info.update(
        statut="estime",
        raison="F classique illustratif ; observations dépendantes possibles.",
        r2=float(model.rsquared),
    )
    coefficients = pd.DataFrame(
        {
            "variable": model.params.index,
            "coefficient": model.params.to_numpy(),
            "ecart_type_groupe": np.nan,
            "p_groupe": np.nan,
        }
    )

    ## Partie 4 - Complément tenant compte des répétitions au sein du joueur
    if info["joueurs"] >= max(20, n_parameters + 2):
        robust = model.get_robustcov_results(
            cov_type="cluster",
            groups=sample["user_id"],
            use_correction=True,
        )
        coefficients["ecart_type_groupe"] = robust.bse
        coefficients["p_groupe"] = robust.pvalues

        # Tester conjointement tous les coefficients sauf la constante.
        restrictions = np.eye(n_parameters)[1:, :]
        covariance = robust.cov_params()
        restricted_covariance = restrictions @ covariance @ restrictions.T
        if np.linalg.matrix_rank(restricted_covariance) == n_parameters - 1:
            global_test = robust.f_test(restrictions)
            info["F_wald_groupes"] = float(np.asarray(global_test.fvalue).item())
            info["p_F_wald_groupes"] = float(np.asarray(global_test.pvalue).item())
        info["raison"] = (
            "F de Wald groupé approximatif ; adversaires communs non corrigés."
        )
    else:
        info["raison"] += " Trop peu de joueurs pour les valeurs p groupées."

    return pd.DataFrame([info]), coefficients, model


In [ ]:
reg_summary, reg_coefficients, model = (
    regression_multiple(clean)
    if RUN_REGRESSIONS
    else (pd.DataFrame([{"statut": "desactivee"}]), pd.DataFrame(), None)
)
display(reg_summary)
display(reg_coefficients)

if model is not None:
    print("Score moyen observé :", sample.score.mean())
    print("R² :", model.rsquared)
    print("Attention : une bonne association ajustée ne démontre pas une cause.")


## Partie 5A Terminer par les descriptifs lisibles
Les descriptifs apparaissent ici en fin de parcours pour faciliter la synthèse ; ils sont aussi à regarder avant d’interpréter les tests. `.mean()` résume le centre ; `.std(ddof=1)` la dispersion ; `.min()` et `.max()` l’étendue ; `.median()` le centre ordonné. L’écart-type des observations n’est pas l’incertitude sur leur moyenne. Avec une seule valeur, l’écart-type d’échantillon est manquant.

La moyenne de `win` donne le taux de victoire. La moyenne de `score` compte les nulles comme un demi-point. La moyenne d’une colonne de répétition 0/1 donne un taux parmi les transitions connues, pas parmi toutes les lignes.


In [ ]:
def statistiques_descriptives(data):
    """Effectif, moyenne, médiane, écart-type, minimum et maximum.

    .std(ddof=1) décrit la dispersion des observations ; ce n'est pas
    l'incertitude sur la moyenne. La moyenne de win est le taux de victoire.
    La moyenne de score compte les nulles comme un demi-point.
    """
    rows = []
    for column in [
        "rating",
        "opponent_rating",
        "elo_diff",
        "score",
        "experience_opening_prior",
    ]:
        values = pd.to_numeric(data[column], errors="coerce")
        rows.append(
            {
                "variable": column,
                "n": values.count(),
                "mean": values.mean(),
                "median": values.median(),
                "std": values.std(ddof=1),
                "min": values.min(),
                "max": values.max(),
            }
        )
    return pd.DataFrame(rows)


In [ ]:
print("Elo moyen :", clean.rating.mean())
print("Écart-type des Elo :", clean.rating.std(ddof=1))
print("Minimum et maximum :", clean.rating.min(), clean.rating.max())
print("Score moyen :", clean.score.mean())
print("Taux de victoire :", clean.win.mean())

display(statistiques_descriptives(clean))
display(clean.groupby("color")["score"].agg(["count", "mean", "std", "min", "max"]))
display(clean.groupby("format_partie")["score"].agg(["count", "mean", "std"]))


## Partie 5B Histogramme et graphiques simples
L’histogramme montre les Elo des parties observées : les joueurs actifs pèsent davantage. Les barres montrent des effectifs de résultats ou un score moyen par couleur. La courbe d’expérience doit être accompagnée de ses effectifs ; elle n’est pas une preuve d’apprentissage causal.


In [ ]:
from analyses.Analyse_ouvertures import exporter_descriptifs
from IPython.display import Image

descriptifs = exporter_descriptifs(clean)
for filename in [
    "histogramme_elo.png",
    "effectifs_resultats.png",
    "score_par_couleur.png",
    "experience_score.png",
]:
    display(Image(filename=str(cfg.RESULTS_DIR / filename)))


## Export complet et interprétation
`analyze` enregistre les parties du parcours dans des fichiers CSV et PNG. On garde les hypothèses, les tailles d’échantillon et les limites. Pour le χ², `valid` vérifie les attentes seulement. Fisher exact, les χ² et les régressions ne suppriment pas automatiquement le biais de sélection ou la dépendance entre parties. Une valeur p manquante signifie souvent que la comparaison n’est pas estimable ; une grande valeur p ne prouve pas l’absence d’association.

Les familles de tests (χ², Fisher exact, régression) sont distinctes. Holm est appliqué séparément aux χ² éligibles et aux quatre Fisher exacts. Les tests globaux de régression ne font pas partie de ces corrections.


In [ ]:
from analyses.Analyse_ouvertures import analyze

tests_generaux = analyze(clean, run_regressions=RUN_REGRESSIONS)
print("Tous les résultats :", cfg.RESULTS_DIR)


In [ ]:
import shutil

archive = shutil.make_archive(str(run_dir), "zip", run_dir)
print("Archive à conserver :", archive)
try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink

    display(FileLink(archive))
else:
    files.download(archive)


## Retrouver les fonctions et les hypothèses
`Nettoyage_donnees.py` prépare les colonnes ; `analyses/Tests_repetition.py` contient le χ² et Fisher exact ; `analyses/Regression_multiple.py` contient le modèle multiple et le F ; `analyses/Analyse_ouvertures.py` suit les cinq parties et produit les descriptifs. Le PDF dans `docs/Guide_pedagogique_analyse.pdf` explique les fichiers et les résultats.

Références : [SciPy χ²](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html), [Fisher exact](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.fisher_exact.html), [loi F](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.f.html), [statsmodels f_test](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.RegressionResults.f_test.html).
